# Comparison: bulletin-fetcher and direct access

Five cases: commerce, environment, agriculture, institutional activity and Spanish search.
The size of the specification, the retrieved acts and metadata and the time of a run are compared.

In [37]:
from pathlib import Path
from collections import Counter
import csv
import re
import sys
import time
import requests
from IPython.display import Markdown, display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/bulletin').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
from bulletin.eurlex.api.client import EurlexBulletinClient

BASE = ROOT / 'scripts/evaluation'
ENDPOINT = 'https://publications.europa.eu/webapi/rdf/sparql'
TIMEOUT = 300
RUN_LIVE = True
client = EurlexBulletinClient(endpoint=ENDPOINT, timeout=TIMEOUT)

## 1. Cases and queries to compare

The files in `queries/` contain saved queries from the library, with the same filters.
If you change a case, also update its `.rq` file. The terms in the title define illustrative samples,
not all acts in each domain. Both methods require Internet access and no credentials.

In [38]:
CASES = [
    {'id': 'trade', 'topic': 'Trade', 'date': '2024-01-01', 'date_end': '2024-03-31', 'language': 'ENG', 'title_contains': 'tariff'},
    {'id': 'environment', 'topic': 'Environment', 'date': '2024-01-01', 'date_end': '2024-03-31', 'language': 'ENG', 'title_contains': 'emission'},
    {'id': 'agriculture', 'topic': 'Agriculture', 'date': '2024-01-01', 'date_end': '2024-03-31', 'language': 'ENG', 'title_contains': 'agricultural', 'category_type': 'REG_IMPL'},
    {'id': 'institution', 'topic': 'Commission announcements', 'date': '2024-01-01', 'date_end': '2024-03-31', 'language': 'ENG', 'category_type': 'ANNOUNC', 'institution_type': 'COM'},
]


PARAMS = {c['id']: {k: v for k, v in c.items() if k not in ('id', 'topic')} for c in CASES}
QUERIES = {c['id']: (BASE / 'queries' / (c['id'] + '.rq')).read_text(encoding='utf-8') for c in CASES}
CALLS = {key: 'client.get_acts(' + ', '.join(f'{k}={v!r}' for k, v in p.items())
         + ", output_format='json')" for key, p in PARAMS.items()}
print(CALLS['agriculture'])
print(QUERIES['agriculture'])

client.get_acts(date='2024-01-01', date_end='2024-03-31', language='ENG', title_contains='agricultural', category_type='REG_IMPL', output_format='json')
PREFIX cdm: <http://publications.europa.eu/ontology/cdm#>
            PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
            PREFIX owl: <http://www.w3.org/2002/07/owl#>
            PREFIX skos: <http://www.w3.org/2004/02/skos/core#>

            SELECT DISTINCT
            ?act
            ?celexAct
            ?actNumber
            ?title
            ?date
            ?sectionCode
            ?subsectionCode
            ?categoryCode
            ?categoryUri
            ?categoryLabel
            ?institutionCode
            ?institutionUri
            ?institutionLabel
            WHERE {
            ?c_act (
                cdm:official-journal-act_date_publication
                | cdm:resource_legal_published_in_official-journal/cdm:publication_general_date_publication
            ) ?date .
            FILTER(?date >= "2024

## 2. Direct access to Cellar

This method uses the SPARQL endpoint of Cellar, with a timeout of 300 seconds per request.

In [39]:
DIRECT_SOURCE = """
FIELDS = dict(zip(
    'act_uri celex_uri act_number title date section_code subsection_code category_code category_uri category_label institution_code institution_uri institution_label'.split(),
    'act celexAct actNumber title date sectionCode subsectionCode categoryCode categoryUri categoryLabel institutionCode institutionUri institutionLabel'.split(),
))

def direct_get_acts(query):
    response = requests.post(ENDPOINT, data={'query': query}, timeout=TIMEOUT,
                             headers={'Accept': 'application/sparql-results+json'})
    response.raise_for_status()
    if response.headers.get('X-SQL-State', '00000') != '00000':
        raise RuntimeError('Cellar devolvió un error o resultados parciales')
    records = [{k: b.get(v, {}).get('value') for k, v in FIELDS.items()}
               for b in response.json()['results']['bindings']]
    for record in records:
        record['date'] = record['date'].split('T')[0]
        record['celex_uri'] = record['celex_uri'] or ''
    return records
""".strip()
exec(DIRECT_SOURCE)

def canonical(records):
    return Counter(tuple(r.get(k) for k in FIELDS) for r in records)

## 3. Comparison

Lexical elements are compared, including the size of the specification, the number of acts and metadata retrieved, and the time taken for a run. The comparison is performed using both the bulletin-fetcher library and direct access to Cellar.


In [40]:
results = []
for index, case in enumerate(CASES):
    key = case['id']
    lib_tokens = len(re.findall(r'\w+', CALLS[key]))
    direct_tokens = len(re.findall(r'\w+', QUERIES[key]))
    row = dict(case=key, library_tokens=lib_tokens, direct_tokens=direct_tokens,
               reduction_pct=round(100 * (1 - lib_tokens / direct_tokens), 2),
               library_acts=None, direct_acts=None, library_seconds=None, direct_seconds=None,
               same_ids=None, same_metadata=None, status='not_run', error='')
    if RUN_LIVE:
        records = {}
        methods = ['library', 'direct'] if index % 2 == 0 else ['direct', 'library']
        for method in methods:
            time.sleep(1)
            start = time.perf_counter()
            try:
                records[method] = (client.get_acts(**PARAMS[key], output_format='json')
                                   if method == 'library' else direct_get_acts(QUERIES[key]))
                row[method + '_acts'] = len(records[method])
            except Exception as exc:
                row['error'] += f'{method}: {exc}; '
            row[method + '_seconds'] = round(time.perf_counter() - start, 3)
        if len(records) == 2:
            a, b = records['library'], records['direct']
            row['same_ids'] = {r['act_uri'] for r in a} == {r['act_uri'] for r in b}
            row['same_metadata'] = canonical(a) == canonical(b)
            row['status'] = ('empty_inconclusive' if not a and not b else
                             'match' if row['same_metadata'] else 'mismatch')
        else:
            row['status'] = 'error'
    results.append(row)

## 4. Results

The results of the comparison are stored in two CSV files: `results/comparison.csv` for the real-time mode and `results/code_metrics.csv` for the offline mode. Each file is overwritten when repeating its mode: it keeps copies if you want to compare executions.

In [41]:
columns = list(results[0])
header = '| ' + ' | '.join(columns) + ' |'
separator = '| ' + ' | '.join(['---'] * len(columns)) + ' |'
rows = [
    '| ' + ' | '.join(str(row[k]).replace('|', '/') for k in columns) + ' |'
    for row in results
]
display(Markdown('\n'.join([header, separator, *rows])))

output = BASE / 'results' / ('comparison.csv' if RUN_LIVE else 'code_metrics.csv')
output.parent.mkdir(exist_ok=True)
with output.open('w', encoding='utf-8', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=columns)
    writer.writeheader()
    writer.writerows(results)
print(output)

| case | library_tokens | direct_tokens | reduction_pct | library_acts | direct_acts | library_seconds | direct_seconds | same_ids | same_metadata | status | error |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| trade | 16 | 390 | 95.9 | 6 | 6 | 0.981 | 1.128 | True | True | match |  |
| environment | 16 | 390 | 95.9 | 12 | 12 | 0.949 | 0.808 | True | True | match |  |
| agriculture | 18 | 394 | 95.43 | 2 | 2 | 1.241 | 1.138 | True | True | match |  |
| institution | 18 | 391 | 95.4 | 171 | 171 | 1.453 | 1.571 | True | True | match |  |

c:\Users\Usuario\Desktop\bulletin-fetcher\scripts\evaluation\results\comparison.csv


## 5. Retrieval and conversion code

Nonblank, non-comment Python source lines and lexical elements (`\w+`) are counted for
the concrete implementations shown here. Both produce lists of dictionaries. The direct baseline
includes the field mapping, request, error checks, response conversion and invocation; the library
provides these operations internally. Imports and client initialization are included.

Scenario parameters, the SPARQL text, shared endpoint/timeout definitions, and evaluation/display/export
code are excluded from both counts. The direct implementation is reusable across all five scenarios:
this is a one-time implementation size, not a cost repeated per case. Counts depend on formatting
and implementation choices and are indicators of exposed code, not measured human effort.

In [42]:
LIBRARY_WORKFLOW = """from bulletin.eurlex.api.client import EurlexBulletinClient
client = EurlexBulletinClient(endpoint=ENDPOINT, timeout=TIMEOUT)
records = client.get_acts(**params, output_format='json')"""
DIRECT_WORKFLOW = 'import requests\n' + DIRECT_SOURCE + '\nrecords = direct_get_acts(query)'

code_results = []
for method, source in [('library', LIBRARY_WORKFLOW), ('direct', DIRECT_WORKFLOW)]:
    lines = [line for line in source.splitlines() if line.strip() and not line.lstrip().startswith('#')]
    code_results.append(dict(method=method, python_lines=len(lines),
                             lexical_elements=len(re.findall(r'\w+', '\n'.join(lines)))))
show_and_save(code_results, 'retrieval_code.csv')
print('Library workflow:\n' + LIBRARY_WORKFLOW)
print('Direct workflow:\n' + DIRECT_WORKFLOW)

method | python_lines | lexical_elements
--- | --- | ---
library | 3 | 19
direct | 18 | 112

c:\Users\Usuario\Desktop\bulletin-fetcher\scripts\evaluation\results\retrieval_code.csv
Library workflow:
from bulletin.eurlex.api.client import EurlexBulletinClient
client = EurlexBulletinClient(endpoint=ENDPOINT, timeout=TIMEOUT)
records = client.get_acts(**params, output_format='json')
Direct workflow:
import requests
FIELDS = dict(zip(
    'act_uri celex_uri act_number title date section_code subsection_code category_code category_uri category_label institution_code institution_uri institution_label'.split(),
    'act celexAct actNumber title date sectionCode subsectionCode categoryCode categoryUri categoryLabel institutionCode institutionUri institutionLabel'.split(),
))

def direct_get_acts(query):
    response = requests.post(ENDPOINT, data={'query': query}, timeout=TIMEOUT,
                             headers={'Accept': 'application/sparql-results+json'})
    response.raise_for_status()
    if response.headers.get('X-SQL-State', '00000') != '00000':
        raise RuntimeError(